# ADMM Convergence Trace Playground

Interactive exploration of ADMM convergence diagnostics —
primal/dual residuals and slack history across iterations.

**Compatible experiments:** `convergence_trace`
(uses `kind == 'trace'` and the `plot_admm_convergence` function).

In [ ]:
# Stage 12: shared setup — make `cordis` importable when this notebook
# is launched from notebooks/, then apply the IEEE paper rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style, load_latest_result, load_run, load_result, summarize,
)

import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH (e.g. on a compute node).
setup_paper_style(use_latex=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

In [ ]:
EXPERIMENT = 'convergence_trace'

# Optional: load a specific run directory instead of the latest.
# Set to a path like 'results/exp_convergence_trace/20260520_113500'
# (relative to the repo root) to re-render an older trace.
EXP_DIR = None

result = load_result(EXPERIMENT, EXP_DIR)
assert result.kind == 'trace'
summarize(result)

# `best_iter` may be 0 (no iterations run) or None (loaded from a
# pre-Stage-18 result that didn't store it).  Cells below handle
# both gracefully.
best_iter = getattr(result.admm_result, 'best_iter', None)
print(f'Best iter: {best_iter}')
print(f'Total iters: {len(result.admm_result.primal_res_history)}')

## 1. Standard convergence plot

In [ ]:
from cordis.plotting import plot_admm_convergence, figsize

# plot_admm_convergence produces a TWO-panel figure (residuals + slack).
# Pre-create both axes and pass them as the `axes=(ax_top, ax_bot)` tuple,
# or call without `axes` to let the function build its own figure.
fig, (ax_res, ax_slack) = plt.subplots(
    1, 2, figsize=figsize(width='double', aspect=2.5/1.5),
)
plot_admm_convergence(result.admm_result, axes=(ax_res, ax_slack))
fig.suptitle('ADMM convergence (primal/dual residuals + slack history)')
plt.tight_layout()
plt.show()

## 2. Primal / dual on twin axes

When primal and dual residuals have very different scales, a single
log-y axis still works but a twin-axes layout makes both visible.
If `best_iter` is set (Stage 18+ trace results), a vertical guide
marks it.

In [ ]:
admm = result.admm_result
iters = np.arange(1, len(admm.primal_res_history) + 1)

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
ln1 = ax.semilogy(iters, admm.primal_res_history, label='primal', color='C0')
ax.set_xlabel('ADMM iteration')
ax.set_ylabel('primal residual', color='C0')
ax.tick_params(axis='y', labelcolor='C0')

ax2 = ax.twinx()
ln2 = ax2.semilogy(iters, admm.dual_res_history, label='dual', color='C3', linestyle='--')
ax2.set_ylabel('dual residual', color='C3')
ax2.tick_params(axis='y', labelcolor='C3')

# Guard against best_iter being None (legacy loaded result) or 0
# (solver returned before recording any iteration).
_bi = getattr(admm, 'best_iter', None)
if _bi:
    ax.axvline(_bi, color='gray', linestyle=':', linewidth=0.8)
    ax.set_title(f'Twin-axes residuals (best iter = {_bi})')
else:
    ax.set_title('Twin-axes residuals (best_iter unavailable)')
plt.show()

## 3. Slack history overlay

When ALM/MoM is used, the slack penalty rises as it discovers QoS
violations.  Overlaying its history on top of residuals shows when
the algorithm 'gives up' enforcing a tight constraint.

In [ ]:
if hasattr(admm, 'slack_history') and admm.slack_history is not None \
        and len(admm.slack_history):
    fig, axes = plt.subplots(2, 1, figsize=figsize(width='single', aspect=3/2.5),
                              sharex=True)
    axes[0].semilogy(iters, admm.primal_res_history, label='primal', color='C0')
    axes[0].semilogy(iters, admm.dual_res_history,   label='dual',   color='C3')
    axes[0].legend()
    axes[0].set_ylabel('residual')
    axes[1].plot(iters, admm.slack_history, color='C2')
    axes[1].set_ylabel('slack')
    axes[1].set_xlabel('ADMM iteration')
    plt.tight_layout()
    plt.show()
else:
    print('No slack_history in this trace.')

## 4. Convergence-rate fit (rough)

Fit a geometric rate to the primal residual tail (last ~30% of iters)
to quantify how fast ADMM is converging.

In [ ]:
tail_frac = 0.3
n         = len(admm.primal_res_history)
tail_start = max(1, int(n * (1 - tail_frac)))
tail_iters = iters[tail_start:]
tail_res   = admm.primal_res_history[tail_start:]

# log y = a + b * iter  →  y = exp(a) * exp(b)^iter
if np.all(tail_res > 0):
    coeffs = np.polyfit(tail_iters, np.log(tail_res), 1)
    rate   = np.exp(coeffs[0])     # geometric ratio per iter
    print(f'Tail geometric rate: r ≈ {rate:.4f}  (per iter)')
    print(f'                     1/(1-r) ≈ {1/(1-rate):.1f}  iters to settle')
else:
    print('Tail contains zeros/negatives; skipping rate fit.')

## 5. Figsize variants

In [ ]:
# Figsize variants — `figsize` returns (w, h) in inches for matplotlib.
# width: 'single' (one column), 'double' (two-column), 'third' (3-up panel).
# aspect: w/h ratio.  Tweak both to fit your paper layout.
from cordis.plotting import figsize

for width in ('single', 'double', 'third'):
    w, h = figsize(width=width, aspect=3/2)
    print(f'{width:>6}: ({w:.2f}, {h:.2f}) inches')

# Example: tight three-up panel for a paper sub-figure
# fig, axes = plt.subplots(1, 3, figsize=figsize(width='double', aspect=3.5/1.5))

## 6. Save with provenance metadata

In [ ]:
# Save with provenance metadata (Git SHA, creation date, etc. — embedded
# into the PDF's metadata, prepended as comments in the .pgf).
from cordis.plotting import save_figure

# Adjust EXPERIMENT and metric labels to match the figure above.
out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_demo',
    formats=('pdf', 'png'),       # add 'pgf' on systems with LaTeX
    metadata={'Experiment': EXPERIMENT, 'Notebook': 'playground'},
)
for p in out:
    print('wrote', p)